# Semantic model sizing pilot
This notebook inventories semantic models visible to the running identity in selected workspaces. It writes only the separate `SemanticModelSizingPilot` Delta table, not `SemanticModels`.

`Estimated_VertiPaq_Bytes` is the dictionary-plus-segment estimate for an Import-only model. `Observed_VertiPaq_Bytes` records a positive response to the same storage query for any mode. For composite models this only describes the local engine storage reported by the query, not upstream models or sources. For models with Direct Lake partitions it is a point-in-time observation, not the size of the underlying OneLake tables, a complete in-memory footprint, or peak memory. DirectQuery does not import the source data. A null is not a zero-size model.

XMLA availability and model permissions are separate from storage mode. Build access is required for XMLA reads, and storage INFO functions can require model write permission. The notebook does not change model permissions, warm Direct Lake with workload queries, or bypass endpoint trust checks.


In [ ]:
%pip install semantic-link-labs==0.14.1


In [ ]:
# Names and IDs are exact, case-insensitive matches. Together they form a union.
# Leave both lists empty to scan all regular workspaces visible to your account.
WORKSPACE_NAMES = [
    'REPLACE_WITH_WORKSPACE_NAME',
]
WORKSPACE_IDS = []
EXCLUDE_WORKSPACE_NAMES = ['Admin monitoring']
EXCLUDE_MODEL_NAMES = ['OneLake Catalog - Governance for Admins', 'Feature Usage and Adoption']
EXCLUDE_MODEL_IDS = []

# [] includes all modes. ['Import'] includes only Import-only models.
# Options: Import, Direct Lake, DirectQuery, Dual, Composite, Unknown.
STORAGE_MODES = []

# Set to None for notebook output only. Otherwise overwrites only the pilot table.
OUTPUT_LAKEHOUSE = 'lh_discovery_v1'


In [ ]:
from collections import Counter
from uuid import UUID

import pandas as pd
import sempy.fabric as fabric
import sempy_labs as labs
from sempy.fabric._client._adomd_connection import FabricAdomdException
from sempy.fabric.exceptions import DatasetNotFoundException, SemPyException, WorkspaceNotFoundException

client = fabric.FabricRestClient()


def pages(path, **filters):
    params = filters.copy()
    while True:
        response = client.get(path, params=params)
        response.raise_for_status()
        page = response.json()
        yield from page['value']
        token = page.get('continuationToken')
        if not token:
            break
        params['continuationToken'] = token


def key(value):
    return value.strip().casefold()


def select_workspaces(workspaces, names, ids, excluded_names):
    wanted_names = {key(name) for name in names}
    wanted_ids = {str(UUID(value.strip())) for value in ids}
    excluded = {key(name) for name in excluded_names}
    name_counts = Counter(key(w['displayName']) for w in workspaces)
    issues = [name for name in wanted_names if name_counts[name] != 1]
    unknown_ids = wanted_ids - {key(w['id']) for w in workspaces}
    if issues or unknown_ids:
        raise ValueError(f'Workspace names missing or ambiguous: {sorted(issues)}; IDs not found: {sorted(unknown_ids)}. Use IDs for duplicate names.')
    return [w for w in workspaces if key(w['displayName']) not in excluded and (
        not (wanted_ids or wanted_names) or key(w['id']) in wanted_ids or key(w['displayName']) in wanted_names
    )]


SIZE_QUERY = '''
EVALUATE ROW (
    "ObservedBytes",
    SUMX ( INFO.STORAGETABLECOLUMNS(), [DICTIONARY_SIZE] )
        + SUMX ( INFO.STORAGETABLECOLUMNSEGMENTS(), [USED_SIZE] )
)
'''


def sizing_errors():
    errors = (SemPyException, FabricAdomdException, ValueError)
    # SemPy loads .NET on demand when it creates an XMLA client.
    try:
        from System import Exception as DotNetException
    except ImportError:
        return errors
    return (*errors, DotNetException)


def probe(call):
    try:
        return call(), None, None
    except (DatasetNotFoundException, WorkspaceNotFoundException) as error:
        return None, 'Not visible to XMLA', str(error).partition('\n')[0][:500]
    except sizing_errors() as error:
        message = str(error)
        untrusted = 'The endpoint for this connection is not a trusted endpoint' in message
        if hasattr(error, 'GetType'):
            namespace = str(error.GetType().Namespace)
            if not (namespace == 'Microsoft.AnalysisServices' or namespace.startswith('Microsoft.AnalysisServices.')
                    or type(error).__name__ == 'ArgumentException' and untrusted):
                raise
        status = type(error).__name__
        if untrusted:
            status = 'Untrusted XMLA endpoint'
        elif 'LargeModelMovedAcrossRegions' in message:
            status = 'Large model moved across regions'
        return None, status, message.partition('\n')[0][:500]


def storage_mode(partitions):
    if partitions is None or 'Mode' not in partitions:
        return 'Unknown', ''
    labels = {'import': 'Import', 'directlake': 'Direct Lake', 'directquery': 'DirectQuery', 'dual': 'Dual'}
    modes = {labels.get(key(str(value)).replace(' ', ''), 'Unknown') for value in partitions['Mode']}
    if not modes or 'Unknown' in modes:
        return 'Unknown', ', '.join(sorted(modes))
    return (next(iter(modes)) if len(modes) == 1 else 'Composite'), ', '.join(sorted(modes))


selected_modes = {key(mode) for mode in STORAGE_MODES}
valid_modes = {'import', 'direct lake', 'directquery', 'dual', 'composite', 'unknown'}
if selected_modes - valid_modes:
    raise ValueError(f'Unsupported storage modes: {sorted(selected_modes - valid_modes)}. Options: {sorted(valid_modes)}')
workspaces = [w for w in pages('/v1/workspaces') if w.get('type') == 'Workspace']
selected = select_workspaces(workspaces, WORKSPACE_NAMES, WORKSPACE_IDS, EXCLUDE_WORKSPACE_NAMES)
excluded_names = {key(name) for name in EXCLUDE_MODEL_NAMES}
excluded_ids = {str(UUID(value.strip())) for value in EXCLUDE_MODEL_IDS}
rows = []
for w in sorted(selected, key=lambda workspace: workspace['displayName'].casefold()):
    workspace_id = UUID(w['id'])
    capacity_id = w.get('capacityId')
    models = list(pages(f'/v1/workspaces/{workspace_id}/items', type='SemanticModel'))
    print(f"{w['displayName']}: {len(models)} visible semantic models", flush=True)
    if not models and not selected_modes:
        rows.append((w['displayName'], str(workspace_id), capacity_id, None, None, None, None,
                     'No visible models', None, None, None))
    for model in models:
        model_id, name = UUID(model['id']), model['displayName']
        identity = (w['displayName'], str(workspace_id), capacity_id, name, str(model_id))
        if str(model_id) in excluded_ids or key(name) in excluded_names:
            if not selected_modes:
                rows.append((*identity, None, None, 'Excluded by configuration', None, None, None))
            continue
        partitions, status, detail = probe(lambda: fabric.list_partitions(dataset=model_id, workspace=workspace_id))
        mode, partition_modes = storage_mode(partitions)
        if selected_modes and key(mode) not in selected_modes:
            continue
        observed = None
        if status is None:
            result, status, detail = probe(lambda: fabric.evaluate_dax(dataset=model_id, workspace=workspace_id, dax_string=SIZE_QUERY))
            if status is None:
                value = None if result.empty else result.iloc[0, 0]
                observed = int(value) if pd.notna(value) and value > 0 else None
                status = 'Positive storage statistics' if observed is not None else 'No positive storage statistics'
        rows.append((*identity, mode, partition_modes, status, observed,
                     observed if mode == 'Import' else None, detail))

schema = ('Workspace_Name string, Workspace_Id string, Capacity_Id string, Model_Name string, Model_Id string, '
          'Storage_Mode string, Partition_Modes string, Size_Status string, Observed_VertiPaq_Bytes long, '
          'Estimated_VertiPaq_Bytes long, Details string')
results = spark.createDataFrame(rows, schema)
if OUTPUT_LAKEHOUSE:
    labs.save_as_delta_table(results, 'SemanticModelSizingPilot', 'overwrite', lakehouse=OUTPUT_LAKEHOUSE,
                             workspace=UUID(notebookutils.runtime.context['currentWorkspaceId']))
display(results.orderBy('Workspace_Name', 'Model_Name'))
display(results.groupBy('Storage_Mode', 'Size_Status').count().orderBy('Storage_Mode', 'Size_Status'))
